# AlpCAN — LIDC-IDRI Veri Keşfi

**Amaç:** LIDC-IDRI veri setini keşfet, nodül dağılımlarını analiz et, model eğitimi için veri yapısını anla.

**Veri Seti:** `zhangweiled/lidcidri` — LIDC-IDRI'den kesilmiş nodül dilimleri (PNG formatı)
- `images/` — BT dilim görüntüleri
- `mask-X/` — 4 radyologun nodül segmentasyon maskeleri

---

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from collections import defaultdict

# Kaggle veri yolu
DATA_DIR = Path("/kaggle/input/lidcidri/LIDC-IDRI-slices")

# Alternatif yolları kontrol et
if not DATA_DIR.exists():
    base = Path("/kaggle/input")
    print("Mevcut input dizinleri:")
    for item in base.iterdir():
        print(f"  {item.name}/")
        for sub in list(item.iterdir())[:5]:
            print(f"    {sub.name}")
    # Dinamik yol bul
    for item in base.rglob("LIDC-IDRI-0001"):
        DATA_DIR = item.parent
        print(f"\nVeri seti bulundu: {DATA_DIR}")
        break
else:
    print(f"Veri seti dizini: {DATA_DIR}")

## 1. Veri Seti Yapısını Keşfet

In [ ]:
# Hasta ve nodül sayılarını topla
patient_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Toplam hasta: {len(patient_dirs)}")
print(f"\nİlk 5 hasta:")
for pdir in patient_dirs[:5]:
    nodules = sorted([d for d in pdir.iterdir() if d.is_dir()])
    print(f"  {pdir.name}: {len(nodules)} nodül")
    for ndir in nodules[:3]:
        contents = list(ndir.iterdir())
        print(f"    {ndir.name}: {[c.name for c in contents]}")

In [ ]:
# Tüm hastaların nodül sayıları
stats = []
total_slices = 0
total_masks = 0

for pdir in patient_dirs:
    nodule_dirs = [d for d in pdir.iterdir() if d.is_dir()]
    for ndir in nodule_dirs:
        img_dir = ndir / "images"
        mask_dirs = [d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")]
        n_slices = len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
        total_slices += n_slices
        total_masks += len(mask_dirs)
        stats.append({
            "patient_id": pdir.name,
            "nodule": ndir.name,
            "n_slices": n_slices,
            "n_annotators": len(mask_dirs),
        })

df = pd.DataFrame(stats)
print(f"\n=== VERİ SETİ ÖZETİ ===")
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"Toplam nodül: {len(df)}")
print(f"Toplam dilim: {total_slices:,}")
print(f"\nHasta başına nodül sayısı:")
print(df.groupby('patient_id').size().describe())
print(f"\nNodül başına dilim sayısı:")
print(df['n_slices'].describe())
print(f"\nAnnotatör dağılımı:")
print(df['n_annotators'].value_counts().sort_index())

## 2. Nodül Görüntü ve Maske Görselleştirme

In [ ]:
# İlk hastanın ilk nodülünü görselleştir
sample_patient = patient_dirs[0]
sample_nodule = sorted([d for d in sample_patient.iterdir() if d.is_dir()])[0]
img_dir = sample_nodule / "images"
mask_dirs = sorted([d for d in sample_nodule.iterdir() if d.is_dir() and d.name.startswith("mask")])

slices = sorted(img_dir.glob("*.png"))
print(f"Hasta: {sample_patient.name}")
print(f"Nodül: {sample_nodule.name}")
print(f"Dilim sayısı: {len(slices)}")
print(f"Annotator sayısı: {len(mask_dirs)}")

# Orta dilimi göster
mid_idx = len(slices) // 2
img = np.array(Image.open(slices[mid_idx]))
print(f"\nGörüntü boyutu: {img.shape}")
print(f"Piksel aralığı: [{img.min()}, {img.max()}]")
print(f"Dtype: {img.dtype}")

In [ ]:
# Görüntü + tüm annotator maskeleri yan yana
n_cols = 1 + len(mask_dirs) + 1  # image + masks + overlay
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
fig.suptitle(f"{sample_patient.name} / {sample_nodule.name} — Dilim {mid_idx}", fontsize=14)

# Orijinal görüntü
axes[0].imshow(img, cmap="gray")
axes[0].set_title("BT Dilimi")
axes[0].axis("off")

# Her annotator maskesi
combined_mask = np.zeros_like(img, dtype=np.float32)
for i, mdir in enumerate(mask_dirs):
    mask_file = mdir / slices[mid_idx].name
    if mask_file.exists():
        mask = np.array(Image.open(mask_file))
        axes[i + 1].imshow(mask, cmap="Reds")
        combined_mask += (mask > 0).astype(np.float32)
    else:
        axes[i + 1].text(0.5, 0.5, "Yok", ha="center", va="center", transform=axes[i + 1].transAxes)
    axes[i + 1].set_title(f"Annotator {i+1}")
    axes[i + 1].axis("off")

# Konsensüs overlay
axes[-1].imshow(img, cmap="gray")
if combined_mask.max() > 0:
    consensus = combined_mask / combined_mask.max()
    axes[-1].imshow(consensus, cmap="jet", alpha=0.5)
axes[-1].set_title(f"Konsensüs ({len(mask_dirs)} annotator)")
axes[-1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/nodule_annotation_preview.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Birden Fazla Nodül Örneği

In [ ]:
# Farklı hastalardan 8 nodül göster
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("LIDC-IDRI — Farklı Hastalardan Nodül Örnekleri", fontsize=14)

sample_indices = np.linspace(0, len(patient_dirs) - 1, 8, dtype=int)

for idx, ax in zip(sample_indices, axes.flat):
    pdir = patient_dirs[idx]
    nodule_dirs = sorted([d for d in pdir.iterdir() if d.is_dir()])
    if not nodule_dirs:
        ax.set_title(f"{pdir.name}: nodül yok")
        ax.axis("off")
        continue
    
    ndir = nodule_dirs[0]
    img_dir = ndir / "images"
    img_files = sorted(img_dir.glob("*.png"))
    if not img_files:
        ax.set_title(f"{pdir.name}: görüntü yok")
        ax.axis("off")
        continue
    
    mid = len(img_files) // 2
    img = np.array(Image.open(img_files[mid]))
    ax.imshow(img, cmap="gray")
    
    # Maske overlay
    mask_dirs_n = [d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")]
    for mdir in mask_dirs_n:
        mask_file = mdir / img_files[mid].name
        if mask_file.exists():
            mask = np.array(Image.open(mask_file))
            if mask.max() > 0:
                ax.contour(mask, colors="red", linewidths=0.5, levels=[0.5])
            break
    
    ax.set_title(f"{pdir.name}\n{ndir.name}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/multi_nodule_preview.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Nodül Boyut Dağılımı Analizi

In [ ]:
# Her nodülün yaklaşık boyutunu hesapla (maske piksel sayısı)
nodule_sizes = []

for i, row in df.iterrows():
    pdir = DATA_DIR / row["patient_id"]
    ndir = pdir / row["nodule"]
    mask_dirs_n = sorted([d for d in ndir.iterdir() if d.is_dir() and d.name.startswith("mask")])
    
    if not mask_dirs_n:
        continue
    
    # İlk annotator maskesinden boyut tahmin et
    total_pixels = 0
    max_diameter = 0
    for mask_file in sorted(mask_dirs_n[0].glob("*.png")):
        mask = np.array(Image.open(mask_file))
        mask_binary = mask > 0
        total_pixels += mask_binary.sum()
        # Bu dilimdeki en büyük çap (piksel)
        if mask_binary.any():
            rows_with = np.any(mask_binary, axis=1)
            cols_with = np.any(mask_binary, axis=0)
            if rows_with.any() and cols_with.any():
                h = np.where(rows_with)[0][-1] - np.where(rows_with)[0][0]
                w = np.where(cols_with)[0][-1] - np.where(cols_with)[0][0]
                max_diameter = max(max_diameter, max(h, w))
    
    nodule_sizes.append({
        "patient_id": row["patient_id"],
        "nodule": row["nodule"],
        "total_mask_pixels": total_pixels,
        "max_diameter_px": max_diameter,
        "n_slices": row["n_slices"],
    })

df_sizes = pd.DataFrame(nodule_sizes)
print(f"Boyut hesaplanan nodül sayısı: {len(df_sizes)}")
print(f"\nMaksimum çap (piksel) istatistikleri:")
print(df_sizes["max_diameter_px"].describe())

In [ ]:
# Boyut dağılımı histogramı
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_sizes["max_diameter_px"], bins=30, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Maksimum Çap (piksel)")
axes[0].set_ylabel("Nodül Sayısı")
axes[0].set_title("Nodül Çap Dağılımı")
axes[0].axvline(x=df_sizes['max_diameter_px'].median(), color='red', linestyle='--', label=f'Medyan: {df_sizes["max_diameter_px"].median():.0f} px')
axes[0].legend()

axes[1].hist(df_sizes["n_slices"], bins=20, edgecolor="black", alpha=0.7, color="coral")
axes[1].set_xlabel("Dilim Sayısı")
axes[1].set_ylabel("Nodül Sayısı")
axes[1].set_title("Nodül Başına Dilim Dağılımı")

plt.tight_layout()
plt.savefig("/kaggle/working/nodule_size_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Konsensüs Maskesi Oluşturma (Ajan 2 Hazırlığı)

In [ ]:
def create_consensus_mask(nodule_dir, threshold=0.5):
    """Birden fazla annotator maskesinden konsensüs maskesi oluştur.
    
    threshold=0.5: En az %50 annotator hemfikir olmalı.
    LIDC-IDRI'de 4 radyolog etiketlemiş — 2/4 veya daha fazlası yeterli.
    """
    mask_dirs = sorted([d for d in nodule_dir.iterdir() if d.is_dir() and d.name.startswith("mask")])
    img_dir = nodule_dir / "images"
    img_files = sorted(img_dir.glob("*.png"))
    
    if not mask_dirs or not img_files:
        return None, None
    
    # İlk görüntüden boyut al
    sample = np.array(Image.open(img_files[0]))
    h, w = sample.shape[:2]
    
    consensus_slices = []
    image_slices = []
    
    for img_file in img_files:
        img = np.array(Image.open(img_file))
        image_slices.append(img)
        
        # Tüm annotator maskelerini topla
        combined = np.zeros((h, w), dtype=np.float32)
        for mdir in mask_dirs:
            mask_file = mdir / img_file.name
            if mask_file.exists():
                mask = np.array(Image.open(mask_file))
                combined += (mask > 0).astype(np.float32)
        
        # Konsensüs: threshold oranında annotator hemfikir
        consensus = (combined / len(mask_dirs)) >= threshold
        consensus_slices.append(consensus.astype(np.uint8))
    
    return np.array(image_slices), np.array(consensus_slices)

# Test et
images, masks = create_consensus_mask(sample_nodule)
if images is not None:
    print(f"Görüntü stack: {images.shape}")
    print(f"Konsensüs maske: {masks.shape}")
    print(f"Pozitif piksel oranı: {masks.sum() / masks.size * 100:.2f}%")

In [ ]:
# Konsensüs maskesi görselleştir
if images is not None:
    n_show = min(len(images), 6)
    fig, axes = plt.subplots(2, n_show, figsize=(3 * n_show, 6))
    fig.suptitle(f"Konsensüs Maskesi — {sample_patient.name}/{sample_nodule.name}", fontsize=13)
    
    indices = np.linspace(0, len(images) - 1, n_show, dtype=int)
    
    for i, idx in enumerate(indices):
        axes[0, i].imshow(images[idx], cmap="gray")
        axes[0, i].set_title(f"Dilim {idx}", fontsize=9)
        axes[0, i].axis("off")
        
        axes[1, i].imshow(images[idx], cmap="gray")
        if masks[idx].max() > 0:
            axes[1, i].contour(masks[idx], colors="lime", linewidths=1.5, levels=[0.5])
        axes[1, i].set_title(f"Konsensüs", fontsize=9)
        axes[1, i].axis("off")
    
    plt.tight_layout()
    plt.savefig("/kaggle/working/consensus_mask_preview.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Özet Rapor ve Sonraki Adımlar

In [ ]:
print("=" * 60)
print("LIDC-IDRI VERİ SETİ ÖZET RAPORU")
print("=" * 60)
print(f"Toplam hasta: {df['patient_id'].nunique()}")
print(f"Toplam nodül: {len(df)}")
print(f"Toplam dilim: {total_slices:,}")
print(f"Ortalama dilim/nodül: {df['n_slices'].mean():.1f}")
print(f"Annotator dağılımı: {dict(df['n_annotators'].value_counts().sort_index())}")

if len(df_sizes) > 0:
    print(f"\nNodül çap (piksel):")
    print(f"  Min: {df_sizes['max_diameter_px'].min():.0f}")
    print(f"  Medyan: {df_sizes['max_diameter_px'].median():.0f}")
    print(f"  Max: {df_sizes['max_diameter_px'].max():.0f}")

print("\n" + "=" * 60)
print("SONRAKİ ADIMLAR:")
print("  1. LUNA16 veri seti ile tam 3D BT analizi")
print("  2. nnU-Net v2 formatına dönüşüm")
print("  3. Konsensüs maskeleri ile eğitim veri seti")
print("  4. 5-fold cross-validation split")
print("  5. CXR pipeline için ChestX-ray14 keşfi")
print("=" * 60)

# Çıktıları kaydet
df.to_csv("/kaggle/working/lidc_nodule_stats.csv", index=False)
if len(df_sizes) > 0:
    df_sizes.to_csv("/kaggle/working/lidc_nodule_sizes.csv", index=False)
print("\nCSV dosyaları kaydedildi.")